# Compare globally aligned short-story result JSON files in CSV

This notebook aligns actions from a baseline and comparison run to the same source story using global verb/argument matching. It writes one CSV with a shared `Sentence` column and one method column per run. Each action record retains its `verb` and `arguments`; a separate diagnostics dataframe records candidate evidence, confidence, and extracted-order inversions.


In [61]:
import json
import math
import re
import sys
from functools import lru_cache
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root():
    """Find the repository root whether Jupyter starts there or in extra/."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data/short_stories").is_dir() and (candidate / "results").is_dir():
            return candidate
    raise RuntimeError("Could not find the llm-action-extraction project root.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from evaluation_helpers import nlp

RESULTS_ROOT = PROJECT_ROOT / "results/short_stories"
STORIES_PATH = PROJECT_ROOT / "data/short_stories/mulab_short_stories/selected_stories.json"

# Change this value to compare the baseline against another result version.
BASELINE_VERSION = "nl2p_1/gpt-5.6-terra"
COMPARISON_VERSION = "nl2p_unfiltered/gpt-5.6-terra"

# Select the story by title so runs with different filename conventions still compare.
STORY_TITLE = "The Three Remarks"
OUTPUT_DIR = Path.cwd() / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / f"{Path(STORY_TITLE).stem}_comparison.csv"


## Global alignment strategy

Each result run is aligned independently to the source story; the resulting action lists are then shown side by side in one sentence dataframe.

- Source sentences and action verbs are normalized by case-folding, splitting punctuation and hyphens, and comparing lemma-like forms. The first verb word must match; every remaining verb word or particle must occur in order within the next five word tokens.
- Every action is compared with every source sentence. Verb evidence generates candidates; there is no fixed sentence lookahead window.
- Arguments are soft disambiguating evidence, not a hard requirement. Exact argument phrases, normalized phrases, and informative argument tokens increase a candidate score, while absent arguments do not reject it.
- A dynamic program selects the highest-scoring action-to-sentence path globally. It prefers textual order, permits repeated actions in one sentence, and permits a small backward movement for genuine local inversions.
- An action with no viable candidate, or whose best global choice is to skip it, is added to the final `Unmatched` row.
- `alignment_diagnostics` records each action's selected sentence, verb and argument evidence, candidate margin, confidence, and order status. An apparent inversion is labelled `candidate_ambiguity` unless both adjacent matches are high-confidence.


In [62]:
WORD_PATTERN = re.compile(r"[A-Za-z0-9]+(?:['’][A-Za-z0-9]+)*")
VERB_CONTEXT_WINDOW = 5
MAX_BACKTRACK_SENTENCES = 2

VERB_EXACT_SCORE = 10.0
VERB_LEMMA_SCORE = 8.0
VERB_PARTICLE_BONUS = 2.0
ARGUMENT_EXACT_SCORE = 4.0
ARGUMENT_NORMALIZED_SCORE = 3.0
ARGUMENT_TOKEN_SCORE = 2.0
ARGUMENT_PARTIAL_SCORE = 1.0
SKIP_ACTION_PENALTY = 6.0
BACKTRACK_PENALTY = 3.0
FORWARD_GAP_PENALTY = 0.15

DETERMINERS = {"a", "an", "the"}
IRREGULAR_LEMMAS = {
    "ate": "eat", "became": "become", "came": "come", "did": "do",
    "fell": "fall", "gave": "give", "gone": "go", "ran": "run",
    "saw": "see", "took": "take", "went": "go", "were": "be",
    "was": "be",
}


def word_tokens(text):
    """Return case-folded tokens, treating hyphens and punctuation as boundaries."""
    return [match.group(0).casefold() for match in WORD_PATTERN.finditer(str(text))]


def simple_stem(word):
    """Provide a conservative fallback form for common English inflections."""
    word = word.casefold()
    if word in IRREGULAR_LEMMAS:
        return IRREGULAR_LEMMAS[word]
    if len(word) > 4 and word.endswith("ies"):
        return word[:-3] + "y"
    if len(word) > 5 and word.endswith("ing"):
        stem = word[:-3]
        return stem[:-1] if len(stem) > 2 and stem[-1:] == stem[-2:-1] else stem
    if len(word) > 4 and word.endswith("ed"):
        stem = word[:-2]
        return stem[:-1] if len(stem) > 2 and stem[-1:] == stem[-2:-1] else stem
    if len(word) > 3 and word.endswith("es"):
        return word[:-2]
    if len(word) > 3 and word.endswith("s"):
        return word[:-1]
    return word


@lru_cache(maxsize=None)
def token_forms(word):
    """Return surface, spaCy lemma, and fallback forms for one normalized token."""
    word = word.casefold()
    parsed = nlp(word)
    spacy_lemma = parsed[0].lemma_.casefold() if parsed else word
    return frozenset({word, spacy_lemma, simple_stem(word)})


def phrase_occurs(phrase_forms, sentence_forms):
    """Return whether a normalized phrase occurs contiguously in a sentence."""
    phrase_length = len(phrase_forms)
    if not phrase_length or phrase_length > len(sentence_forms):
        return False
    return any(
        all(
            phrase_forms[offset] & sentence_forms[start + offset]
            for offset in range(phrase_length)
        )
        for start in range(len(sentence_forms) - phrase_length + 1)
    )


def ordered_tail_occurs(tail_words, sentence_words, leading_index):
    """Match verb particles or remaining verb words within the local word window."""
    if not tail_words:
        return True
    window = sentence_words[
        leading_index + 1 : leading_index + 1 + VERB_CONTEXT_WINDOW
    ]
    window_index = 0
    for tail_word in tail_words:
        try:
            matched_offset = window.index(tail_word, window_index)
        except ValueError:
            return False
        window_index = matched_offset + 1
    return True


def verb_match_details(action_verb, sentence_features):
    """Return normalized verb-match evidence or None for a sentence."""
    verb_words = word_tokens(action_verb)
    if not verb_words:
        return None

    leading_word, *tail_words = verb_words
    leading_forms = token_forms(leading_word)
    best_match = None
    for index, sentence_word in enumerate(sentence_features["words"]):
        if not ordered_tail_occurs(tail_words, sentence_features["words"], index):
            continue
        if sentence_word == leading_word:
            match_type = "exact"
            score = VERB_EXACT_SCORE
        elif leading_forms & sentence_features["forms"][index]:
            match_type = "lemma"
            score = VERB_LEMMA_SCORE
        else:
            continue
        if tail_words:
            score += VERB_PARTICLE_BONUS
        candidate = {
            "verb_match_type": match_type,
            "verb_score": score,
            "trigger_token_index": index,
        }
        if best_match is None or candidate["verb_score"] > best_match["verb_score"]:
            best_match = candidate
    return best_match


def argument_match_score(argument, sentence_features):
    """Return soft evidence that an argument helps identify a sentence."""
    argument_words = word_tokens(argument)
    if not argument_words:
        return 0.0, "none"
    argument_forms = [token_forms(word) for word in argument_words]
    if all(
        word in sentence_features["words"]
        for word in argument_words
    ) and phrase_occurs(
        [frozenset({word}) for word in argument_words], sentence_features["surface_forms"]
    ):
        return ARGUMENT_EXACT_SCORE, "exact_phrase"
    if phrase_occurs(argument_forms, sentence_features["forms"]):
        return ARGUMENT_NORMALIZED_SCORE, "normalized_phrase"

    informative_forms = [
        forms
        for word, forms in zip(argument_words, argument_forms)
        if word not in DETERMINERS
    ]
    if not informative_forms:
        return 0.0, "none"
    sentence_form_union = set().union(*sentence_features["forms"])
    matched = sum(bool(forms & sentence_form_union) for forms in informative_forms)
    if matched == len(informative_forms):
        return ARGUMENT_TOKEN_SCORE, "all_tokens"
    if matched:
        return ARGUMENT_PARTIAL_SCORE * matched / len(informative_forms), "partial_tokens"
    return 0.0, "none"


def build_sentence_features(sentences):
    """Precompute normalized lexical features for all source sentences."""
    features = []
    for index, sentence in enumerate(sentences):
        words = word_tokens(sentence)
        features.append(
            {
                "index": index,
                "text": sentence,
                "words": words,
                "surface_forms": [frozenset({word}) for word in words],
                "forms": [token_forms(word) for word in words],
            }
        )
    return features


def normalize_action(action_index, action):
    """Validate an action while retaining empty argument placeholders."""
    if not isinstance(action, dict) or not str(action.get("verb", "")).strip():
        raise ValueError(f"Action {action_index} has no valid verb: {action!r}")
    arguments = action.get("arguments", [])
    if not isinstance(arguments, list) or not all(
        isinstance(argument, str) for argument in arguments
    ):
        raise ValueError(f"Action {action_index} has invalid arguments: {arguments!r}")
    return {"verb": str(action["verb"]), "arguments": arguments}


def score_action_sentence(action, sentence_features):
    """Score one action/sentence pair; return None when its verb has no match."""
    verb_details = verb_match_details(action["verb"], sentence_features)
    if verb_details is None:
        return None
    argument_details = [
        argument_match_score(argument, sentence_features)
        for argument in action["arguments"]
    ]
    argument_score = (
        sum(score for score, _ in argument_details) / len(argument_details)
        if argument_details
        else 0.0
    )
    return {
        "sentence_index": sentence_features["index"],
        "verb_match_type": verb_details["verb_match_type"],
        "verb_score": verb_details["verb_score"],
        "trigger_token_index": verb_details["trigger_token_index"],
        "argument_score": argument_score,
        "argument_match_types": [match_type for _, match_type in argument_details],
        "score": verb_details["verb_score"] + argument_score,
    }


def build_candidate_lattice(sentences, actions):
    """Build globally available, verb-grounded sentence candidates for every action."""
    sentence_features = build_sentence_features(sentences)
    normalized_actions = [
        normalize_action(index, action) for index, action in enumerate(actions)
    ]
    lattice = [
        [
            candidate
            for sentence_feature in sentence_features
            if (candidate := score_action_sentence(action, sentence_feature)) is not None
        ]
        for action in normalized_actions
    ]
    return normalized_actions, lattice


def order_penalty(frontier, sentence_index):
    """Prefer order while allowing small, explicitly penalized local inversions."""
    if frontier < 0 or sentence_index == frontier:
        return 0.0
    if sentence_index > frontier:
        return FORWARD_GAP_PENALTY * math.log1p(sentence_index - frontier)
    return BACKTRACK_PENALTY * (frontier - sentence_index)


def global_align_actions_to_sentences(sentences, actions):
    """Choose the globally highest-scoring, near-monotonic action/sentence path."""
    normalized_actions, lattice = build_candidate_lattice(sentences, actions)
    # A state is keyed by the furthest source sentence reached. Its choices contain
    # one candidate dict or None for each processed action.
    states = {-1: (0.0, [])}
    for candidates in lattice:
        next_states = {}
        for frontier, (path_score, choices) in states.items():
            skipped = (path_score - SKIP_ACTION_PENALTY, choices + [None])
            if frontier not in next_states or skipped[0] > next_states[frontier][0]:
                next_states[frontier] = skipped
            for candidate in candidates:
                sentence_index = candidate["sentence_index"]
                if sentence_index < frontier - MAX_BACKTRACK_SENTENCES:
                    continue
                next_frontier = max(frontier, sentence_index)
                candidate_score = (
                    path_score
                    + candidate["score"]
                    - order_penalty(frontier, sentence_index)
                )
                proposed = (candidate_score, choices + [candidate])
                if (
                    next_frontier not in next_states
                    or proposed[0] > next_states[next_frontier][0]
                ):
                    next_states[next_frontier] = proposed
        states = next_states

    _, best_choices = max(states.values(), key=lambda state: state[0])
    aligned = [[] for _ in sentences]
    unmatched = []
    diagnostics = []
    previous_choice = None
    for action_index, (action, candidates, choice) in enumerate(
        zip(normalized_actions, lattice, best_choices)
    ):
        local_scores = sorted(
            (candidate["score"] for candidate in candidates), reverse=True
        )
        if choice is None:
            unmatched.append(action)
            diagnostics.append(
                {
                    "Action index": action_index,
                    "Verb": action["verb"],
                    "Arguments": action["arguments"],
                    "Sentence index": None,
                    "Verb match": "none",
                    "Verb score": 0.0,
                    "Argument score": 0.0,
                    "Candidate score": None,
                    "Candidate count": len(candidates),
                    "Score margin": None,
                    "Confidence": "unmatched",
                    "Order status": "unmatched",
                }
            )
            continue

        aligned[choice["sentence_index"]].append(action)
        alternative_scores = [
            candidate["score"]
            for candidate in candidates
            if candidate["sentence_index"] != choice["sentence_index"]
        ]
        second_score = max(alternative_scores, default=None)
        score_margin = (
            choice["score"] - second_score if second_score is not None else choice["score"]
        )
        if (
            choice["verb_match_type"] == "exact"
            and (choice["argument_score"] >= ARGUMENT_PARTIAL_SCORE or score_margin >= 2.0)
        ):
            confidence = "high"
        elif score_margin >= 1.0:
            confidence = "medium"
        else:
            confidence = "low"

        if previous_choice is None:
            order_status = "first_match"
        else:
            difference = choice["sentence_index"] - previous_choice["sentence_index"]
            if difference > 0:
                order_status = "ordered"
            elif difference == 0:
                order_status = "same_sentence"
            elif (
                previous_choice["confidence"] == "high"
                and confidence == "high"
            ):
                order_status = "local_inversion"
            else:
                order_status = "candidate_ambiguity"

        diagnostic = {
            "Action index": action_index,
            "Verb": action["verb"],
            "Arguments": action["arguments"],
            "Sentence index": choice["sentence_index"],
            "Verb match": choice["verb_match_type"],
            "Verb score": choice["verb_score"],
            "Argument score": choice["argument_score"],
            "Candidate score": choice["score"],
            "Candidate count": len(candidates),
            "Score margin": score_margin,
            "Confidence": confidence,
            "Order status": order_status,
        }
        diagnostics.append(diagnostic)
        previous_choice = {"sentence_index": choice["sentence_index"], "confidence": confidence}

    return aligned, unmatched, pd.DataFrame(diagnostics)


def summarize_order_diagnostics(diagnostics):
    """Summarize grounded order quality without treating low-confidence cases as errors."""
    matched = diagnostics.dropna(subset=["Sentence index"])
    status_counts = matched["Order status"].value_counts()
    comparable = len(matched) - int((matched["Order status"] == "first_match").sum())
    inversions = int(status_counts.get("local_inversion", 0))
    ambiguities = int(status_counts.get("candidate_ambiguity", 0))
    return {
        "matched": len(matched),
        "unmatched": len(diagnostics) - len(matched),
        "comparable": comparable,
        "ordered_or_same": int(status_counts.get("ordered", 0) + status_counts.get("same_sentence", 0)),
        "local_inversions": inversions,
        "candidate_ambiguities": ambiguities,
    }


METHOD_COLUMN_NAMES = {
    "nl2p_1": "nl2p",
    "nl2p_unfiltered_simple": "nl2p unfiltered simple",
}


def result_column_name(version):
    """Return a method-only action-column name for one result version."""
    method = Path(version).parts[0]
    return METHOD_COLUMN_NAMES.get(method, method.replace("_", " "))


def load_result(results_dir, story_title):
    """Load the unique single-story result for a title from one result version."""
    results_dir = Path(results_dir)
    if not results_dir.is_dir():
        raise ValueError(f"Result directory does not exist: {results_dir}")

    matches = []
    for input_path in results_dir.glob("*.json"):
        if input_path.name == "results.json":
            continue
        with input_path.open(encoding="utf-8") as file:
            result = json.load(file)
        if (
            isinstance(result, dict)
            and isinstance(result.get("actions"), list)
            and str(result.get("dataset_title", "")).casefold() == story_title.casefold()
        ):
            matches.append((input_path, result))

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one per-story result for {story_title!r} in "
            f"{results_dir}; found {len(matches)}."
        )
    return matches[0]


def load_source_story(result):
    """Find the unique source story associated with a result object."""
    with STORIES_PATH.open(encoding="utf-8") as file:
        stories = json.load(file)["stories"]

    content_hash = result.get("content_sha256")
    matches = [story for story in stories if story.get("content_sha256") == content_hash]
    if len(matches) != 1:
        dataset_title = result.get("dataset_title")
        matches = [story for story in stories if story.get("dataset_title") == dataset_title]
    if len(matches) != 1:
        raise ValueError(
            "Expected exactly one source story for "
            f"{result.get('dataset_title')!r}; found {len(matches)}."
        )
    return matches[0]


def split_sentences(content):
    """Split source content into non-empty sentences with spaCy."""
    sentences = [sentence.text.strip() for sentence in nlp(content).sents if sentence.text.strip()]
    if not sentences:
        raise ValueError("The source story did not contain any sentences.")
    return sentences


def comparison_json_to_dataframe(
    baseline_version, comparison_version, story_title, return_diagnostics=False
):
    """Build globally aligned sentence rows for the baseline and comparison runs."""
    baseline_path, baseline_result = load_result(RESULTS_ROOT / baseline_version, story_title)
    comparison_path, comparison_result = load_result(
        RESULTS_ROOT / comparison_version, story_title
    )
    baseline_story = load_source_story(baseline_result)
    comparison_story = load_source_story(comparison_result)
    if baseline_story["content"] != comparison_story["content"]:
        raise ValueError(
            "The baseline and comparison results resolve to different source stories: "
            f"{baseline_path} and {comparison_path}."
        )

    sentences = split_sentences(baseline_story["content"])
    baseline_actions, baseline_unmatched, baseline_diagnostics = global_align_actions_to_sentences(
        sentences, baseline_result["actions"]
    )
    comparison_actions, comparison_unmatched, comparison_diagnostics = global_align_actions_to_sentences(
        sentences, comparison_result["actions"]
    )
    baseline_column = result_column_name(baseline_version)
    comparison_column = result_column_name(comparison_version)
    rows = [
        {
            "Sentence": sentence,
            baseline_column: baseline_sentence_actions,
            comparison_column: comparison_sentence_actions,
        }
        for sentence, baseline_sentence_actions, comparison_sentence_actions in zip(
            sentences, baseline_actions, comparison_actions
        )
    ]
    if baseline_unmatched or comparison_unmatched:
        rows.append(
            {
                "Sentence": "Unmatched",
                baseline_column: baseline_unmatched,
                comparison_column: comparison_unmatched,
            }
        )
    dataframe = pd.DataFrame(
        rows, columns=["Sentence", baseline_column, comparison_column]
    )

    baseline_diagnostics.insert(0, "Method", baseline_column)
    comparison_diagnostics.insert(0, "Method", comparison_column)
    diagnostics = pd.concat(
        [baseline_diagnostics, comparison_diagnostics], ignore_index=True
    )
    if return_diagnostics:
        return dataframe, diagnostics
    return dataframe


def comparison_json_to_csv(
    baseline_version, comparison_version, story_title, output_csv=None
):
    """Globally align two result versions and save their shared Sentence/NL2P CSV."""
    dataframe, diagnostics = comparison_json_to_dataframe(
        baseline_version, comparison_version, story_title, return_diagnostics=True
    )
    output_path = (
        Path(output_csv)
        if output_csv is not None
        else PROJECT_ROOT / "extra" / f"{Path(story_title).stem}_nl2p_comparison.csv"
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    csv_dataframe = dataframe.copy()
    for column in csv_dataframe.columns.drop("Sentence"):
        csv_dataframe[column] = csv_dataframe[column].map(
            lambda actions: json.dumps(actions, ensure_ascii=False)
        )
    csv_dataframe.to_csv(output_path, index=False)
    return dataframe, diagnostics, output_path


def action_counts(dataframe, action_column):
    """Return aligned and unmatched action counts for one dataframe column."""
    unmatched_mask = dataframe["Sentence"].eq("Unmatched")
    unmatched_count = (
        len(dataframe.loc[unmatched_mask, action_column].iloc[0])
        if unmatched_mask.any()
        else 0
    )
    action_count = sum(len(actions) for actions in dataframe[action_column])
    return action_count - unmatched_count, unmatched_count


def display_wrapped_dataframe(dataframe):
    """Display fully wrapped dataframe values in Jupyter."""
    with pd.option_context("display.max_colwidth", None):
        display(
            dataframe.style.set_properties(
                **{
                    "white-space": "normal",
                    "overflow-wrap": "anywhere",
                    "vertical-align": "top",
                    "text-align": "left",
                }
            )
        )


In [63]:
comparison_dataframe, alignment_diagnostics, saved_csv = comparison_json_to_csv(
    BASELINE_VERSION, COMPARISON_VERSION, STORY_TITLE, OUTPUT_CSV
)
baseline_column = result_column_name(BASELINE_VERSION)
comparison_column = result_column_name(COMPARISON_VERSION)
baseline_aligned, baseline_unmatched = action_counts(
    comparison_dataframe, baseline_column
)
comparison_aligned, comparison_unmatched = action_counts(
    comparison_dataframe, comparison_column
)
sentence_row_count = len(comparison_dataframe) - int(
    comparison_dataframe["Sentence"].eq("Unmatched").any()
)
print(
    f"Saved {sentence_row_count} sentence rows to {saved_csv}\n"
    f"{baseline_column}: {baseline_aligned} aligned actions, "
    f"{baseline_unmatched} unmatched actions\n"
    f"{comparison_column}: {comparison_aligned} aligned actions, "
    f"{comparison_unmatched} unmatched actions"
)
for method, method_diagnostics in alignment_diagnostics.groupby("Method", sort=False):
    summary = summarize_order_diagnostics(method_diagnostics)
    print(
        f"{method} order diagnostics: {summary['ordered_or_same']} ordered/same, "
        f"{summary['local_inversions']} local inversions, "
        f"{summary['candidate_ambiguities']} candidate ambiguities"
    )
display_wrapped_dataframe(comparison_dataframe)
display_wrapped_dataframe(alignment_diagnostics)


Saved 86 sentence rows to /home/char37/Projects/MPhil/llm-action-extraction/short_stories_expt/output/The Three Remarks_comparison.csv
nl2p: 47 aligned actions, 0 unmatched actions
nl2p unfiltered: 77 aligned actions, 1 unmatched actions
nl2p order diagnostics: 46 ordered/same, 0 local inversions, 0 candidate ambiguities
nl2p unfiltered order diagnostics: 76 ordered/same, 0 local inversions, 0 candidate ambiguities


,Sentence,nl2p,nl2p unfiltered
0,"There was once a princess, the most beautiful princess that ever was seen.",[],[]
1,"Her hair was black and soft as the raven's wing; her eyes were like stars dropped in a pool of clear water, and her speech like the first tinkling cascade of the baby Nile.",[],[]
2,"She was also wise, graceful, and gentle, so that one would have thought she must be the happiest princess in the world.",[],[]
3,"But, alas!",[],[]
4,there was one terrible drawback to her happiness.,[],[]
5,She could make only three remarks.,[],[]
6,"No one knew whether it was the fault of her nurse, or a peculiarity born with her; but the sad fact remained, that no matter what was said to her, she could only reply in one of three phrases.",[],[]
7,"The first was, ""What is the price of butter?""",[],[]
8,"The second, ""Has your grandmother sold her mangle yet?""",[],[]
9,"And the third, ""With all my heart!""",[],[]


,Method,Action index,Verb,Arguments,Sentence index,Verb match,Verb score,Argument score,Candidate score,Candidate count,Score margin,Confidence,Order status
0,nl2p,0,knelt,[],16.000000,exact,10.000000,0.000000,10.000000,1,10.000000,high,first_match
1,nl2p,1,hung up,['their armor'],19.000000,exact,12.000000,4.000000,16.000000,1,16.000000,high,ordered
2,nl2p,2,took,['the vows'],19.000000,lemma,8.000000,4.000000,12.000000,5,2.000000,medium,same_sentence
3,nl2p,3,flogging,['themselves'],19.000000,exact,10.000000,4.000000,14.000000,1,14.000000,high,same_sentence
4,nl2p,4,wearing,['hair shirts'],19.000000,exact,10.000000,4.000000,14.000000,1,14.000000,high,same_sentence
5,nl2p,5,putting,['dry toast-crumbs'],19.000000,exact,10.000000,4.000000,14.000000,3,6.000000,high,same_sentence
6,nl2p,6,slipped down,[],25.000000,exact,12.000000,0.000000,12.000000,1,12.000000,high,ordered
7,nl2p,7,opened,['the back door'],25.000000,exact,10.000000,4.000000,14.000000,4,4.000000,high,same_sentence
8,nl2p,8,ran away,[],25.000000,exact,12.000000,0.000000,12.000000,1,12.000000,high,same_sentence
9,nl2p,9,wandered,[],26.000000,exact,10.000000,0.000000,10.000000,1,10.000000,high,ordered
